# Table of Contents

* [Problem Introduction](#problem-introduction)
    * [Lit Review](#lit-review)
* [Dataset Creation](#dataset-creation)
    * [Process](#process)
    * [runtrack](#runtrack)
* [Model Formulation](#model-formulation)
    * [Description](#description)
    * [Mathematical formulation](#mathematical-formulation)
    * [Code](#code)
* [Instances](#instances)
    * [Graph sizes](#graph-sizes)
    * [Run lengths](#run-lengths)
    * [Generated instances](#generated-instances)
* [Results](#results)
* [Improvements](#improvements)
    * [Interior node removal](#interior-node-removal)
    * [Undirected graph](#undirected-graph)
    * [Failed attempts:](#failed-attempts)
        * [Penalty constraint](#penalty-constraint)
        * [Lower bound](#lower-bound)
        * [Branching](#branching)
* [Conclusion](#conclusion)
* [Declaration of Contribution](#declaration-of-contribution)
* [References](#references)

# Problem Introduction

## Literature Review

# Dataset Creation

## Process

## runtrack

# Model Formulation

## Description

Let $G(V, A)$ denote a directed graph with vertices $V$ and arcs $A$. Each arc $a \in A$ has an associated profit, $s_{ij}$, historic penalty $q_{ij}$, current penalty, $p_{ij}$, and length, $c_{ij}$. The profit of an arc is derived from the CityStrides data and the penalties are derived from the visited count of that particular arc. Choosing to run an arc will incurr a cost comprising of its profit and penalties. The final route must form a tour, where the solution must start and end at the depot, and must not contain any subtours. The total distance must be within 80%-100% of the user-defined desired run length. The objective is to maximize the total profit incurred by a tour. 


## Mathematical formulation

The decision variables are $x_{ij}, y_{ij}$ and $z_{ij}$, and are defined below: 

\begin{align*}
    x_{ij} = \text{Number of times arc $(i,j)$ is ran}
\end{align*}

\begin{align*}
y_{ij} = 
\begin{cases}
1 & \text{if arc $(i,j)$ is chosen to be ran}  \\
0 & \text{otherwise }
\end{cases}
\end{align*}

\begin{align*}
    z_{ij} = 
    \begin{cases}
    1 & \text{if either arc $(i,j)$ or arc $(j,i)$ is chosen to be ran}  \\
    0 & \text{otherwise }
    \end{cases}
\end{align*}

The variables $x_{ij}$ and $y_{ij}$ are used to define choosing an arc $(i,j)$ to be ran. The variable $z_{ij}$ is used to determine the profit of an arc. The profit of running a particular arc is independent of the direction, ie. running an arc $(i,j)$ from $i$ to $j$ should incur the same profit as $j$ to $i$ and should only be incurred once if both directions are ran. Therefore, the variable $z_{ij}$ ensures that if we run an arc $(i,j)$ at least once in either direction, we incur the profit of the arc. 

The full model formulation is presented below:

\begin{align*}
    \max & \sum_{(i,j)\in A} s_{ij}z_{ij} - q_{ij}y_{ij} -p_{ij}x_{ij} && \tag{1.1}\\
    \text{s.t.} &  \sum\limits_{j \in V\setminus i} x_{ij}  = \sum\limits_{j \in V\setminus i} x_{ji} && \forall i \in V & \tag{1.2}\\
    & z_{ij} \geq y_{ij} && \forall (i,j) \in A, i\leq\ j & \tag{1.3}\\
    & z_{ij} \geq y_{ji} && \forall (i,j) \in A, i\leq j & \tag{1.4}\\
    & y_{ij} + y_{ji} \geq z_{ij} && \forall (i,j) \in A, i\leq j & \tag{1.5}\\
    & \sum\limits_{i \in V\setminus S, j \in S} x_{ij} \geq y_{kl} && \forall S \subset V\setminus \{0\},  \forall(k,l) \in A(S) & \tag{1.6}\\
    & x_{ij} \geq y_{ij} && \forall (i,j) \in A & \tag{1.7}\\
    & y_{ij}\cdot 2 \geq x_{ij} && \forall (i,j) \in A &\tag{1.8} \\
    & \sum\limits_{(i,j) \in A} c_{ij}x_{ij} \leq T_{max} & \tag{1.9}\\
    & \sum\limits_{(i,j) \in A} c_{ij}x_{ij} \geq 0.8\cdot T_{max} & \tag{1.10}\\
    & \sum\limits_{j\in A(0)} y_{0j} \geq 1 & \tag{1.11}\\
    & x_{ij} \in \mathbb{N} && \forall (i,j) \in A & \tag{1.12}\\
    & y_{ij} \in \{0,1\} && \forall (i,j) \in A & \tag{1.13}\\
    & z_{ij} \in \{0,1\} && \forall (i,j) \in A, i<j & \tag{1.14}\\
\end{align*}

The Objective function (1.1) defines the total profit of a run and is accumulated from the profit of running new arcs minus the historic and current penalty from running arcs that have already been ran. Constraint (1.2) is the symmetry constraint which ensure flow conservation at each node. The set of Constraints (1.3-1.5) reflect the bi-conditional relationship between the variables $y_{ij}$ and $z_{ij}$, where the variable $z_{ij}$ must be 1 if either $y_{ij}$ or $y_{ji}$ is 1, and 0 otherwise. Subtours are eliminated by Constraint (1.6), where $S$ denotes any tour that does not contain the depot 0 and therefore must be a subtour. The constraint eliminates these subtours by ensuring that for all edges in the subtour $y \in S$, you must enter the subtour at least once using edges between nodes that are in the subtour, $j \in S$, and the nodes that are not in the subtour, $i \in S$. Constraints (1.7-1.8) reflect the bi-conditional relationship between the variables $x_{ij}$ and $y_{ij}$ where $y_{ij}=1$ if $x_{ij}>=1$ and vice versa. The total length is reflected in the Constraints (1.9-1.10) where the total distance ran must be less than the maximimum $T_{max}$ but more than 80% of $T_{max}$. The second constraint of these two was added to ensure that the final route is still close to our desired run length. Without it, the model can return a short route that maximimizes the profit by not running edges that have been already ran. Finally, Constraint (1.11) ensures that we start at the depot, which is node 0, by forcing the model to leave the depot at least once. 

## Code

# Instances

## Graph sizes

* jess-min
* jess
* jin

## Run lengths

* 5K
* 10K
* 15K

## Generated instances

# Results

# Improvements

## Interior node removal

## Undirected graph

## Failed attempts:

### Penalty constraint

### Lower bound

### Branching

# Conclusion

# Declaration of Contribution

# References